In [0]:
from pyspark.sql.functions import col, explode

GREENHOUSE_RAW_PATH = (
    "s3://job-market-analysis-pipeline/"
    "raw-data/greenhouse/"
    "ingestion_date=2026-10-06/*.json"
)

greenhouse_raw = spark.read.json(GREENHOUSE_RAW_PATH)

print("Raw files loaded successfully")
print("Raw records:", greenhouse_raw.count())

Raw files loaded successfully
Raw records: 5


In [0]:
greenhouse_bronze = (
    greenhouse_raw
    .select(
        col("source"),
        col("company"),
        col("board_token"),
        col("ingestion_timestamp"),
        col("ingestion_date"),
        explode(col("jobs")).alias("job")
    )
    .select(
        col("source"),
        col("company"),
        col("board_token"),
        col("ingestion_timestamp"),
        col("ingestion_date"),

        col("job.id").alias("job_id"),
        col("job.title").alias("job_title"),
        col("job.location").alias("location"),
        col("job.departments").alias("departments"),
        col("job.offices").alias("offices"),
        col("job.metadata").alias("metadata"),
        col("job.content").alias("content"),
        col("job.employment").alias("employment"),
        col("job.education").alias("education"),
        col("job.first_published").alias("first_published"),
        col("job.updated_at").alias("updated_at"),
        col("job.absolute_url").alias("job_url")
    )
)

display(greenhouse_bronze.limit(10))

source company board_token ingestion_timestamp ingestion_date job_id job_title location departments offices metadata content employment education first_published updated_at job_url greenhouse Cloudflare cloudflare 2026-10-06T12:44:37.740098+00:00 2026-10-06 7695702 Account Executive, FedCiv List(Hybrid) List(List(List(), 70660, Field Sales, 29066)) List(List(List(), 19994, Washington, DC, United States, Washington, DC, 48752)) List(List(825338, Cost Center, 5150 - Field Sales, single_select), List(8654220, Job Posting Location, ["Washington DC, US"], multi_select), List(46546702, Career Site Department, ["Sales"], multi_select)) <div class="content-intro"><h3>About Us</h3>
<p>At Cloudflare, we are on a mission to help build a better Internet. Today the company runs one of the world’s largest networks that powers millions of websites and other Internet properties for customers ranging from individual bloggers to SMBs to Fortune 500 companies. Cloudflare protects and accelerates any Internet application online without adding hardware, installing software, or changing a line of code. Internet properties powered by Cloudflare all have web traffic routed through its intelligent global network, which gets smarter with every request. As a result, they see significant improvement in performance and a decrease in spam and other attacks. Cloudflare was named to Entrepreneur Magazine’s Top Company Cultures list and ranked among the World’s Most Innovative Companies by Fast Company.</p>
<p>At Cloudflare, we’re not looking for people who wait for a polished roadmap; we’re looking for the builders who see the cracks in the Internet that everyone else has simply learned to live with. We value candidates who have the instinct to spot a "normalized" problem and the AI-native curiosity to create a solution using the latest tools. Our culture is built on iteration, leveraging AI to ship faster today to make it better tomorrow, while ensuring that every improvement, no matter how small, is shared across the team to lift everyone up. If you’re the type of person who values curiosity over bureaucracy, and that AI is a partner in solving tough problems to keep the Internet moving forward, you’ll fit right in.</p></div><h2>Available Locations</h2>
<ul>
<li>Washington DC, US</li>
</ul>
<h2>About the Role</h2>
<p>Account Executives, Channel Account Managers, Business Development Representatives, Solution Engineers, Customer Success, and Sales Operations - all working together help our customers adopt Cloudflare and create great Internet-enabled experiences. The sales team at Cloudflare helps customers solve real, technical problems while creating the revenue streams that help the company provide free service to millions in our community.</p>
<h2>Responsibilities</h2>
<ul>
<li>We are looking for a seasoned sales professional to help cover the Federal market for US. The ideal candidate will possess both a sales and technical background that enables them to drive engagement at all levels with technical and non-technical buyers.</li>
<li>As an Account Executive, you'll be responsible for developing and executing against a comprehensive account/territory plan as well as driving sales in a defined territory and/or account list to achieve revenue targets. The person in this role will also assist in creating and articulating compelling value propositions for Cloudflare services.</li>
<li>Manage contract negotiations</li>
<li>Maintain a robust sales pipeline</li>
<li>Develop long-term strategic relationships with key accounts.</li>
</ul>
<h2>Desirable Skills, Knowledge, and Experience</h2>
<ul>
<li>Minimum 5 years of experience selling technical solutions or products to the Public Sector (Federal Government) ; <strong>HHS, SSA or DOT</strong> experience preferred (but open to other agency experience)</li>
<li>Experience managing longer, complex sales cycles`</li>
<li>Basic understanding of computer networking and “how the internet works”</li>
<li>Aptitude for l

In [0]:
print("Greenhouse Bronze records:", greenhouse_bronze.count())

Greenhouse Bronze records: 1583


In [0]:
greenhouse_bronze.printSchema()

root
 |-- source: string (nullable = true)
 |-- company: string (nullable = true)
 |-- board_token: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- job_id: long (nullable = true)
 |-- job_title: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |-- departments: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- child_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- parent_id: long (nullable = true)
 |-- offices: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- child_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- location: string (nullable = true)
 | 

In [0]:
from pyspark.sql.functions import count, countDistinct

greenhouse_bronze.select(
    count("*").alias("total_records"),
    countDistinct("job_id").alias("unique_job_ids")
).show()

+-------------+--------------+
|total_records|unique_job_ids|
+-------------+--------------+
|         1583|          1583|
+-------------+--------------+



In [0]:
greenhouse_bronze.groupBy("company").count().orderBy("company").show()

+-------------------+-----+
|            company|count|
+-------------------+-----+
|         Cloudflare|  416|
|           Coinbase|  226|
|              Figma|  161|
|GuidePoint Security|   61|
|             Stripe|  719|
+-------------------+-----+



In [0]:
(
    greenhouse_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("job_market.bronze.greenhouse_jobs")
)

In [0]:
spark.sql("""
    SELECT *
    FROM job_market.bronze.greenhouse_jobs
    LIMIT 10
""").display()

source company board_token ingestion_timestamp ingestion_date job_id job_title location departments offices metadata content employment education first_published updated_at job_url greenhouse Cloudflare cloudflare 2026-10-06T12:44:37.740098+00:00 2026-10-06 7695702 Account Executive, FedCiv List(Hybrid) List(List(List(), 70660, Field Sales, 29066)) List(List(List(), 19994, Washington, DC, United States, Washington, DC, 48752)) List(List(825338, Cost Center, 5150 - Field Sales, single_select), List(8654220, Job Posting Location, ["Washington DC, US"], multi_select), List(46546702, Career Site Department, ["Sales"], multi_select)) <div class="content-intro"><h3>About Us</h3>
<p>At Cloudflare, we are on a mission to help build a better Internet. Today the company runs one of the world’s largest networks that powers millions of websites and other Internet properties for customers ranging from individual bloggers to SMBs to Fortune 500 companies. Cloudflare protects and accelerates any Internet application online without adding hardware, installing software, or changing a line of code. Internet properties powered by Cloudflare all have web traffic routed through its intelligent global network, which gets smarter with every request. As a result, they see significant improvement in performance and a decrease in spam and other attacks. Cloudflare was named to Entrepreneur Magazine’s Top Company Cultures list and ranked among the World’s Most Innovative Companies by Fast Company.</p>
<p>At Cloudflare, we’re not looking for people who wait for a polished roadmap; we’re looking for the builders who see the cracks in the Internet that everyone else has simply learned to live with. We value candidates who have the instinct to spot a "normalized" problem and the AI-native curiosity to create a solution using the latest tools. Our culture is built on iteration, leveraging AI to ship faster today to make it better tomorrow, while ensuring that every improvement, no matter how small, is shared across the team to lift everyone up. If you’re the type of person who values curiosity over bureaucracy, and that AI is a partner in solving tough problems to keep the Internet moving forward, you’ll fit right in.</p></div><h2>Available Locations</h2>
<ul>
<li>Washington DC, US</li>
</ul>
<h2>About the Role</h2>
<p>Account Executives, Channel Account Managers, Business Development Representatives, Solution Engineers, Customer Success, and Sales Operations - all working together help our customers adopt Cloudflare and create great Internet-enabled experiences. The sales team at Cloudflare helps customers solve real, technical problems while creating the revenue streams that help the company provide free service to millions in our community.</p>
<h2>Responsibilities</h2>
<ul>
<li>We are looking for a seasoned sales professional to help cover the Federal market for US. The ideal candidate will possess both a sales and technical background that enables them to drive engagement at all levels with technical and non-technical buyers.</li>
<li>As an Account Executive, you'll be responsible for developing and executing against a comprehensive account/territory plan as well as driving sales in a defined territory and/or account list to achieve revenue targets. The person in this role will also assist in creating and articulating compelling value propositions for Cloudflare services.</li>
<li>Manage contract negotiations</li>
<li>Maintain a robust sales pipeline</li>
<li>Develop long-term strategic relationships with key accounts.</li>
</ul>
<h2>Desirable Skills, Knowledge, and Experience</h2>
<ul>
<li>Minimum 5 years of experience selling technical solutions or products to the Public Sector (Federal Government) ; <strong>HHS, SSA or DOT</strong> experience preferred (but open to other agency experience)</li>
<li>Experience managing longer, complex sales cycles`</li>
<li>Basic understanding of computer networking and “how the internet works”</li>
<li>Aptitude for l

In [0]:
spark.sql("""
    SELECT
        company,
        COUNT(*) AS job_count
    FROM job_market.bronze.greenhouse_jobs
    GROUP BY company
    ORDER BY job_count DESC
""").display()

company,job_count
Stripe,719
Cloudflare,416
Coinbase,226
Figma,161
GuidePoint Security,61
